# Asthma ER Visit Rates by County

### Libraries

In [ ]:
from pathlib import Path
import pandas as pd

### Settings

In [ ]:
DATA = Path("/Volumes/01-2TB/NJ-Truck-and-Warhouse-Pollution/data")  # external SSD
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
CORE = ["Essex", "Middlesex", "Union"]

## Load asthma ER visits

In [ ]:
asthma = pd.read_excel(RAW / "health" / "asthma_dataset.xlsx")
asthma.columns = ["group", "county", "year", "ed_visits", "footnote"]
asthma = asthma[asthma["year"].notna() & (asthma["county"] != "Unknown Residence")]
asthma["year"] = asthma["year"].astype(int)
asthma["ed_visits"] = pd.to_numeric(asthma["ed_visits"].astype(str).str.replace(",", ""), errors="coerce")
asthma = asthma[["county", "year", "ed_visits"]].reset_index(drop=True)

print(f"{asthma['county'].nunique()} counties, {asthma['year'].min()}-{asthma['year'].max()}, "
      f"{asthma['ed_visits'].isna().sum()} suppressed values")
asthma.head()

## Load population and join

In [ ]:
population = pd.read_csv(RAW / "census" / "nj_county_population.csv", dtype={"county_fips": str})

rates = asthma.merge(population, on=["county", "year"], how="left", validate="one_to_one")
assert rates["population"].notna().all(), rates.loc[rates["population"].isna(), ["county", "year"]]

rates["rate_per_10k"] = (rates["ed_visits"] / rates["population"] * 10_000).round(1)
rates["covid_year"] = rates["year"] == 2020
rates.head()

## Core corridor counties

In [ ]:
rates[rates["county"].isin(CORE)].pivot(index="county", columns="year", values="rate_per_10k")

## All counties, ranked

In [ ]:
summary = (rates[~rates["covid_year"]]
           .groupby("county")
           .agg(avg_rate_per_10k=("rate_per_10k", "mean"), avg_visits=("ed_visits", "mean"))
           .round(1)
           .sort_values("avg_rate_per_10k", ascending=False))
summary["rank"] = range(1, len(summary) + 1)
summary["core_county"] = summary.index.isin(CORE)
summary

## Save

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
out = PROCESSED / "asthma_county_rates.csv"
rates[["county", "county_fips", "year", "ed_visits", "population", "rate_per_10k", "covid_year"]].to_csv(out, index=False)
print(f"Saved {len(rates)} county-years to {out.relative_to(DATA.parent)}")